### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ghanas_indigenous_intel",
    dataset_year="2025",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="Zindi",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/phantom50/ghanas-indigenous-intel-dataset",
    download_description=r"""
We download the data from Kaggle, although it was originally a Zindi competition.

kaggle datasets download phantom50/ghanas-indigenous-intel-dataset \
&& unzip ghanas-indigenous-intel-dataset.zip \
&& rm ghanas-indigenous-intel-dataset.zip \
&& mkdir -p local-data-warehouse/ghanas_indigenous_intel \
&& mv *.csv local-data-warehouse/ghanas_indigenous_intel/
""",
    # References
    academic_reference_bibtex=r"""@misc{zindi_ghana_indigenous_intel_2025,
    author       = {{Zindi}},
    title        = {Ghana's Indigenous Intel Challenge [BEGINNERS ONLY]: Data},
    year         = {2025},
    howpublished = {\url{https://zindi.africa/competitions/ghana-indigenous-intel-challenge/data}},
    note         = {Zindi dataset page. Accessed 2026-04-11}
}
""",
    academic_reference_bibtex_key="zindi_ghana_indigenous_intel_2025",
    license="CC-BY-SA 4.0",
    data_tags=["Non-IID", "Temporal", "Grouped"],
    curation_comments="""
- Information about the data source: The data was collected by local farmers in the Pra River Basin of Ghana using the Smart Indigenous Weather App, a custom-built mobile app for data collection. Farmers made weather forecasts based on traditional signs such as cloud formations, sun position, wind, moon, heat, and specific tree or animal behaviors. The task is to predict the farmers' forecasts based on these traditional signs, which are represented as features in the dataset. The target variable is the actual amount of rain in four categories, therefore the actual task is to make a prediction correcting the farmers' forecasts.
- The data was initially used in a Zindi competition with a temporal split, we do the same.
- We create 6 splits using a sliding window approach with a window size of 3 days and a step size of 2 days. This means that one day overlaps among consecutive test splits.
- We ensure that each split uses at least half of the available samples for training.
- We drop the ID column.
- We convert prediction_time to datetime and sort by it.
- We rename the target column to "rainfall" for better clarity.
- We transform object columns and user_id to categorical.

- Note: The split that would be ideal for the task would be grouped+temporal. However, the available data is not enough for this scenario. In general the available data size reduces the task quality a lot.
- Anomaly: For this dataset it is possible that test folds don't contain some classes.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="rainfall",
    problem_type="multiclass_classification",
    objective_metric_name="f1_macro",
    stratify_on="rainfall",
    time_on="prediction_time",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
test_df = pd.read_csv(dataset_mold.path / "test.csv")

df = df.drop(columns=["ID"])
test_df = test_df.drop(columns=["ID"])

df["prediction_time"] = pd.to_datetime(df["prediction_time"])
test_df["prediction_time"] = pd.to_datetime(test_df["prediction_time"])

df = df.rename(columns={"Target": "rainfall"})

cat_cols = ['user_id', 'community', 'district', 'indicator', 'indicator_description', 'time_observed', 'rainfall']
for col in cat_cols:
    df[col] = df[col].astype("category")

print("Loaded data shape:", df.shape)

Loaded data shape: (10928, 11)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    target_feature=task_mold.target_column_name,
    problem_type=task_mold.problem_type,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,928
Columns: 11
Use sampling: False (sample size: 10,928)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['prediction_time', 'user_id', 'community', 'indicator_description', 'indicator', 'time_observed', 'predicted_intensity', 'district', 'confidence', 'forecast_length']
Rows remaining as candidates after top-10 filter: 0 (of 10,928)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,user_id,confidence,predicted_intensity,community,district,prediction_time,indicator,indicator_description,time_observed,rainfall,forecast_length
0,11,0.3,0.0,Tumfa,atiwa_west,2025-05-30 11:09:33,NaN,NaN,NaN,MEDIUMRAIN,12
1,17,0.3,0.0,Kwabeng,atiwa_west,2025-05-30 11:09:35,NaN,NaN,NaN,HEAVYRAIN,12
2,19,0.3,0.0,Akropong,atiwa_west,2025-05-30 11:09:47,NaN,NaN,NaN,MEDIUMRAIN,12
3,23,0.3,0.0,Asamama,atiwa_west,2025-05-30 11:16:33,NaN,NaN,NaN,HEAVYRAIN,12
4,23,0.3,0.0,Asamama,atiwa_west,2025-05-30 11:16:55,NaN,NaN,NaN,HEAVYRAIN,12


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,time_observed,category,10856.0,99.34,8.0,"EVENING, EARLY_MORNING, MORNING, ALL_DAY, AFTERNOON, LATE_AFTERNOON, NIGHT, DAWN"
1,indicator_description,category,10582.0,96.83,28.0,"Clouds moving South → East, Cloud (probably cumulus) without visible sky, Gathering of clouds in the east, Fast-moving Cirrus clouds (white, visible at dawn), Sunny with clouds, Light fog, Heated atmosphere, Sharp moving clouds (cirrus) with visible stars, High intensity sun, Heavy Clouds (probably cumulus) reflecting rays of sunlight"
2,indicator,category,10425.0,95.40,10.0,"clouds, sun, heat, fog, wind, moon, dew, star, thunder, lightning"
3,user_id,category,0.0,0.00,43.0,"18, 47, 23, 66, 27, 53, 62, 22, 7, 81"
4,community,category,0.0,0.00,38.0,"Akwaduuso, FOSO ODUMASI , Asamama, Assin nyankomasi , Akropong , Tumfa, Kwabeng , Assin ATONSU , Awenare, Abomosu"
5,district,category,0.0,0.00,3.0,"atiwa_west, assin_fosu, obuasi_east"
6,rainfall,category,0.0,0.00,4.0,"NORAIN, MEDIUMRAIN, HEAVYRAIN, SMALLRAIN"
7,prediction_time,datetime64[ns],0.0,0.00,10887.0,"2025-07-02 06:51:01, 2025-07-02 06:51:00, 2025-07-02 06:50:59, 2025-06-19 08:38:36, 2025-07-12 19:18:07, 2025-07-02 06:50:58, 2025-07-18 17:45:31, 2025-06-13 07:14:51, 2025-07-14 21:21:12, 2025-07-08 18:25:00"
8,confidence,float64,0.0,0.00,3.0,"0.3, 0.6, 1.0"
9,predicted_intensity,float64,0.0,0.00,4.0,"0.0, 0.66, 0.33, 1.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
confidence,10928.0,0.540474,0.272068,0.3,1.0
predicted_intensity,10928.0,0.026830,0.132468,0.0,1.0
forecast_length,10928.0,19.135432,5.891856,12.0,24.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                rank                                                       
community             1                                              Akwaduuso   
                      2                                          FOSO ODUMASI    
                      3                                                Asamama   
                      4                                      Assin nyankomasi    
                      5                                              Akropong    
district              1                                             atiwa_west   
                      2                                             assin_fosu   
                      3                                            obuasi_east   
indicator             1                                                   <NA>   
                      2                                                 clouds   
                      3                                                    sun   
                      4                                                   heat   
                      5                                                    fog   
indicator_description 1                                                   <NA>   
                      2                             Clouds moving South → East   
                      3           Cloud (probably cumulus) without visible sky   
                      4                        Gathering of clouds in the east   
                      5     Fast-moving Cirrus clouds (white, visible at dawn)   
prediction_time       1                                    2025-07-02 06:51:01   
                      2                                    2025-07-02 06:51:00   
                      3                                    2025-07-02 06:50:59   
                      4                                    2025-06-19 08:38:36   
                      5                                    2025-07-12 19:18:07   
rainfall              1                                                 NORAIN   
                      2                                             MEDIUMRAIN   
                      3                                              HEAVYRAIN   
                      4                                              SMALLRAIN   
time_observed         1                                                   <NA>   
                      2                                                EVENING   
                      3                                          EARLY_MORNING   
                      4                                                MORNING   
                      5                                                ALL_DAY   
user_id               1                                                     18   
                      2                                                     47   
                      3                                                     23   
                      4                                                     66   
                      5                                                     27   

                            count    pct  
column                rank                
community             1      1427  13.06  
                      2      1179  10.79  
                      3      1139  10.42  
                      4       853   7.81  
                      5       668   6.11  
district              1      4877  44.63  
                      2      4815  44.06  
                      3      1236  11.31  
indicator             1     10425  95.40  
                      2       266   2.43  
                      3        90   0.82  
                      4        53   0.48  
                      5        27   0.25  
indicator_description 1     10582  96.83  
                      2        52   0.48  
                      3        45   0.41  
                      4        29   0.27  
                      5        24   0.22  
prediction_tim

In [8]:
# Target Distribution
target_df

,count,pct
rainfall,,
NORAIN,9612,87.96
MEDIUMRAIN,761,6.96
HEAVYRAIN,315,2.88
SMALLRAIN,240,2.20


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

target_col = task_mold.target_column_name

day_of_year = df["prediction_time"].dt.dayofyear

window = 3
step = 2

max_day = day_of_year.max()
last_start = max_day - window + 1
min_train_size = len(df) / 2

# walk backward from the last valid split start
valid_starts = []
split = last_start

while split >= day_of_year.min():
    train_size = (day_of_year < split).sum()
    if train_size < min_train_size:
        break
    valid_starts.append(split)
    split -= step

valid_starts = sorted(valid_starts)

used_in_train = set()
used_in_test = set()
used_data = set()
splits = {}
for s_num, split in enumerate(valid_starts[::-1]):
    train_idx = day_of_year[day_of_year < split].index
    test_idx = day_of_year[(day_of_year >= split) & (day_of_year < split + window)].index

    train_size = (day_of_year < split).sum()
    test_size = ((day_of_year >= split) & (day_of_year < split + window)).sum()

    splits[s_num] = {0: [train_idx.tolist(), test_idx.tolist()]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(
        f"Train size (days < {split}) = {train_size}, "
        f"Test size (days {split}-{split + window - 1}) = {test_size}"
    )
    print("Train target dist:", df.loc[train_idx, target_col].value_counts(normalize=True))
    print("Test target dist:", df.loc[test_idx, target_col].value_counts(normalize=True))

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(splits)} splits created.")
print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create 6 splits using a sliding window approach with a window size of 3 days and a step size of 2 days. This means that one day overlaps among consecutive test splits. We ensure that each split uses at least half of the available samples for training. ",
    splits=splits,
    time_horizon=3,
    time_horizon_unit="days",
)

Train size (days < 199) = 9933, Test size (days 199-201) = 995
Train target dist: rainfall
NORAIN        0.876271
MEDIUMRAIN    0.067955
HEAVYRAIN     0.031712
SMALLRAIN     0.024061
Name: proportion, dtype: float64
Test target dist: rainfall
NORAIN        0.912563
MEDIUMRAIN    0.086432
SMALLRAIN     0.001005
HEAVYRAIN     0.000000
Name: proportion, dtype: float64
Train size (days < 197) = 8609, Test size (days 197-199) = 1792
Train target dist: rainfall
NORAIN        0.858985
MEDIUMRAIN    0.077942
HEAVYRAIN     0.036590
SMALLRAIN     0.026484
Name: proportion, dtype: float64
Test target dist: rainfall
NORAIN        0.971540
MEDIUMRAIN    0.022321
SMALLRAIN     0.006138
HEAVYRAIN     0.000000
Name: proportion, dtype: float64
Train size (days < 195) = 7506, Test size (days 195-197) = 1705
Train target dist: rainfall
NORAIN        0.839861
MEDIUMRAIN    0.088463
HEAVYRAIN     0.041966
SMALLRAIN     0.029710
Name: proportion, dtype: float64
Test target dist: rainfall
NORAIN        0.988

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to ghanas_indigenous_intel/019dd375-4259-77c1-a177-5acc1f4d0134
019dd375-4259-77c1-a177-5acc1f4d0134
d47199f80bd667dacc88b8dd87bf8b5cd6d7892fd9bbcfba5002a87d8e7b2d36
